# Python Bootcamp Day 4

## Lecture 1: Multi-dimensional Data with Xarray (Continued)

### Lesson Goals
By the end of this lecture, you will be able to:
- Recall the core building blocks of xarray (Datasets, DataArrays, coordinates, dimensions) from yesterday's lesson
- Select and subset multi-dimensional data efficiently using `slice`
- Compute averages and sums across dimensions with `.mean()` and `.sum()`
- Use `.groupby()` to split data into groups (e.g., by season) and compute group-wise statistics
- Understand the difference between weighted and unweighted averages, and why it matters for unevenly-spaced time groups like months within a season
- Convert between xarray, numpy, and pandas data structures depending on the task at hand

### Outline
1. **Quick Review from Yesterday** — Datasets vs. DataArrays, indexing with `.isel()`
2. **2.4 Manipulating Multi-Dimensional Arrays**
   - `slice` and `mean`: selecting a time period and averaging over it
   - `groupby`: grouping data by season (DJF, MAM, JJA, SON)
   - Weighted vs. unweighted seasonal averages
3. **2.5 Xarray to NumPy** — converting DataArrays/Datasets to NumPy arrays
4. **2.6 Xarray to Pandas** — converting DataArrays to pandas Series/DataFrames

In [ ]:
# Mount the github repo to access data
from google.colab import drive
drive.mount('/content/drive')

# Path to shared bootcamp data (repo-root Datasets/)
DATA_FOLDER = "/content/drive/MyDrive/python_bootcamp/python_bootcamp_for_earth_science/Datasets/"

### Quick Review from Yesterday

In [ ]:
# Path to shared bootcamp data (repo-root Datasets/)
DATA_FOLDER = "../../Datasets/"

Import the packages and libraries

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

Read in the data using `open_dataset`

In [ ]:
ds = xr.open_dataset(DATA_FOLDER + 'precip.mon.mean.nc')

In [ ]:
ds # run this cell and click around below to get familiar with xarray dataset structure

<font color="#0F766E">Question: What are the 4 different categories available in each `xarray` dataset?</font>

<font color="#0F766E">Question: What is the code below doing?</font>

In [ ]:
ds['precip'].isel(lat=40).isel(lon=70)

In [ ]:
ds['precip'].isel(lat=40).isel(lon=70).plot()

Let's move on to the second half of the `xarray` lesson for today.

<font color="#0F766E">Question: What is the code below doing differently than the above?</font>

In [ ]:
ds['precip'].sel(lat=-11.25, lon=176.25)

In [ ]:
ds['precip'].sel(lat=-11.25, lon=176.25).plot()

<font color="#0F766E">Question: What is different here?</font>

In [ ]:
ds['precip'].where((ds.lat>10) & (ds.lon<90))

In [ ]:
ax = plt.axes(projection=ccrs.PlateCarree())
ds['precip'].where((ds.lat>10) & (ds.lon<90)).isel(time=0).plot(transform=ccrs.PlateCarree())
ax.coastlines()
plt.show()

<font color="#0F766E">Question: What is different between these two approaches below?</font>

In [ ]:
%%time
ds['precip'].where((ds.lat>10) & (ds.lon<90)).mean() # approach 1

In [ ]:
%%time
ds['precip'].where((ds.lat>10) & (ds.lon<90), drop=True).mean() # approach 2

The `where` function applies a mask to the dataset. You can speed up the computation by dropping the NaNs in the dataset by toggling "drop" 

In [ ]:
ds['precip'][0,:,:].plot()

In [ ]:
ds['precip'].where(ds.precip>20)[0,:,:].plot()

In [ ]:
ds['precip'].where(ds.precip>20,drop=True).mean('time')

How has the dimensions of the dataset changed with this most recent masking based on precipitation amount?

### 2.4 Manipulating multi-dimensional arrays

Tools we will use:
* `slice`
* `mean`
* `sum`
* `groupby`

For more information on xarray tools: http://xarray.pydata.org/en/stable/user-guide/index.html
<br/>For more information on xarray indexing: http://xarray.pydata.org/en/stable/user-guide/indexing.html

For convenience, we will store the precipitation data as it's own data array.

Notice that we are sub-selecting an xarray DataArray from an xarray Dataset

In [ ]:
precip = ds.precip
precip

### `slice` and `mean` Example 

We can use `slice` to select only a small portion of the data.

Why use `slice`? So that it is faster and easier to deal with. For instance, if you had 1 million data points for each year from the year 2010 - 2020 (so, total of 10 million data points), and you were only interested in the year 2004, you can use `slice` to select the year 2004 data to work on.

We can use `.mean` to average the data. Let's look at the average precipitation in 1995.

**Goal**: select 1995 dates, January to December, and take the mean over the dimension of `time`, then calculate and plot the result.

Let's do this step-by-step. The last line of the code cell below is a concise way of combining multiple operations in one line. Which do you prefer? And why?

In [ ]:
# Step 1: select the time slice for 1995 and store it in a new variable called `precip_1995`
precip_1995 = precip.sel(time=slice('1995-01-01','1995-12-01'))

# Step 2: take the mean over the dimension of `time` and store it in a new variable called `precip_1995_mean`
precip_1995_mean = precip_1995.mean(dim='time')

# Step 3: plot the mean precipitation for 1995 and store the plot in a variable called `precip_1995_plot`
precip_1995_plot = precip_1995_mean.plot()

In [ ]:
### alternative code that does the same thing but is more concise:
# notice how we can chain the operations together without needing to create intermediate variables
precip_1995_plot = precip.sel(time=slice('1995-01-01','1995-12-01')).mean(dim='time').plot()

### `groupby` Example 

`.groupby` is another useful tool. It allows us to split the data into multiple groups, work within these groups, and combine them back into a single dataset. Let's group time by season:

In [ ]:
precip.groupby('time.season')

The data has been divided into four seasons: 

'DJF' for December-January-February (winter)

'JJA' for June-July-August (summer)

'MAM' for March-April-May (spring)

'SON' for September-October-November (fall). 

These are common shorthand notations used in Earth science.

Using `.sum`, we can sum over a given dimension. Combining groupby and sum, we can calculate the weighted seasonal averages. Let's just pick one year and one grid for simplicity. The equation below shows an example on how to calculate the weighted averaged preipitation for spring (MAM).

$$
P_{\text{spring}}
=
P_{\text{Mar}} \cdot \frac{31}{31+30+31}
+
P_{\text{Apr}} \cdot \frac{30}{31+30+31}
+
P_{\text{May}} \cdot \frac{31}{31+30+31}
$$

### Calculate the spring precipitation rate in 1995

Start by identifying the time period and location of interest.

We will now store this as a separate variable called `precip_example` and we will use Jan 1 -- Dec 31, 1995 as the time range. We will also select a grid point for this example.

In [ ]:
# select the time slice for 1995, then select the latitude of 1.25 and longitude of 72, and store the result in a new variable called `precip_example`
# you can come back and change the latitude and longitude to see how the results change for different locations
precip_example = precip.sel(time=slice('1995-01-01','1995-12-31')).sel(lat=1.25).sel(lon=72, method='nearest')

View the example data

In [ ]:
precip_example

<font color="#0F766E">Question: What is `nearest` doing here? Run the cell above and see what would happen if you removed it.</font>

In [ ]:
# run this to show how longitude is stored in the dataset and why we need to use the `method='nearest'` argument in the previous cell
precip.lon

Ensure that we have the right days per month and that all 12 months are present in the data

In [ ]:
month_length = precip_example.time.dt.days_in_month
month_length

In [ ]:
# Calculate the weights by grouping by 'time.season' since each month/season has a different number of days, we need to weight the average by the number of days in each month.
# In this case, we are calculating the weights for each month based on the number of days in that month. We then group by season and divide by the sum of days in that season to get the normalized weights.
weights = month_length.groupby('time.season') / month_length.groupby('time.season').sum()

# Calculate the weighted average by multiplying the original data by the weights and then summing over the 'time' dimension (instead of taking the mean). This will give us the weighted average precipitation for each season.
precip_weighted = (precip_example * weights).groupby('time.season').sum(dim='time')
precip_weighted

In [ ]:
# Compared with unweighted seasonal averages by directly taking the mean over the 'time' dimension after grouping by 'time.season'
precip_unweighted = precip_example.groupby('time.season').mean(dim='time')
precip_unweighted

<font color="#0F766E"> Calculate the sum of precipitation during the spring (over March, April, and May) of 1995. </font><br>
Remember, the equation is:
$$
P_{\text{spring}}
=
P_{\text{Mar}} \cdot \frac{31}{31+30+31}
+
P_{\text{Apr}} \cdot \frac{30}{31+30+31}
+
P_{\text{May}} \cdot \frac{31}{31+30+31}
$$

In [ ]:
# Calculate the sum of precipitation over March, April, and May of 1995 using our example variable.
precip_spring_weighted = precip_weighted.sel(season='MAM')
precip_spring_unweighted = precip_unweighted.sel(season='MAM')

In [ ]:
print('Spring 1995 precipitation rate (weighted):', precip_spring_weighted.values)
print('Spring 1995 precipitation rate (unweighted):', precip_spring_unweighted.values)

<font color="#0F766E"> What are the units? </font><br>

Another `groupby` example - calculating anomalies  
In the geosciences, we typically think of anomalies as the value of something (e.g. precipitation)  
relative to the normal values for that time of year.  
Let's calculate anomalies of precipitation, where anomaly = value - normal values for that calendar month

In [ ]:
precip_climatology = precip.sel(lat=1.25).sel(lon=72, method='nearest').groupby("time.month").mean()
precip_example_anom = precip_example.groupby("time.month") - precip_climatology
precip_example_anom

### 2.5 Xarray to numpy

Once again, numpy is a powerful tool. Therefore, it is to our benefit to be able to convert xarray datasets to numpy arrays. 
<br/> We can do this with `np.array()`.

In [ ]:
lat    = np.array(ds['lat'])
lon    = np.array(ds['lon'])
time   = np.array(ds['time'])
precip = np.array(ds['precip'])
lat

As seen above, Xarray introduces labels on top of raw NumPy-like multidimensional arrays, which allows for a more intuitive, more concise, and less error-prone developer experience. For example, with label names, you’ll easily understand what you were thinking when you come back to look at your code weeks or months later.

Notice also how numpy requires you to create an individual variable for each of the data categories while xarray allowed you to use a single dataset where all the data was self-contained.

Xarray also allows you to convert it to different formats using other ways.

<font color="#0F766E"> See what happens when you use `.values` and `.data` (Example: `ds['precip'].values`) </font><br>

In [ ]:
ds['precip'].values

### 2.6 Xarray to pandas

Xarray and pandas objects are also interchangeable. A shortcut to convert a dataArray directly into a pandas object is  `DataArray.to_pandas()` (i.e., a 1D array is converted to a `Series`, 2D to `DataFrame` and 3D to `Panel`).

We would like to save the global average rainfall into an excel file. However, before we calculate the global average, we must weight the data by latitude

In [ ]:
weights = np.cos(np.deg2rad(ds.lat))
weights.name = "weights"
weights
weighted_precip = ds.weighted(weights)
weighted_mean = weighted_precip.mean(("lon","lat"))

In [ ]:
#this visualizes the weighting effect as a function of latitude
(weights*ds.lon.where(ds.lon>360,1)).plot(cbar_kwargs={"label":"weighting"})

Now let's compare the timeseries of the weighted global average to an unweighthed average (variable called "unweighted_mean"). 

Calculate a timeseries of the global average rainfall using the `mean` function. Then plot both timeseries.

How different are they?

In [ ]:
#your code goes here

Convert these two timeseries into pandas dataframes, which we can then use to export as excel files or CSVs.

In [ ]:
unweighted_mean_df = unweighted_mean.to_pandas()
weighted_mean_df = weighted_mean.to_pandas()

In [ ]:
weighted_mean_df=weighted_mean_df.rename(columns={"precip":"precip_weighted"})
weighted_mean_df["precip_unweighted"]=unweighted_mean_df.precip

In [ ]:
weighted_mean_df

Now you can export this dataframe to excel using `to_excel`

Xarray is heavily inspired by pandas, which also works with labeled arrays, and it uses pandas internally. The main distinguishing feature of xarray's dataarray over labeled arrays in pandas is that dimensions can have names (e.g. "time", "latitude"), which are easier to keep track of than axis numbers. While pandas is a great tool for working with tabular data, it can get a little awkward when data is of higher dimension (See more at https://docs.xarray.dev/en/stable/).